In [1]:
import pandas as pd 

df = pd.read_parquet('datasets/transformed_sample_dataset_6m.parquet')
df.head()

,event_timestamp,location_id,item_id,quantity,cluster,region,is_holiday
0,2017-01-01,25,99197,1.0,1,Santa Elena,0.0
1,2017-01-01,25,103665,7.0,1,Santa Elena,0.0
2,2017-01-01,25,105574,1.0,1,Santa Elena,0.0
3,2017-01-01,25,105857,4.0,1,Santa Elena,0.0
4,2017-01-01,25,106716,2.0,1,Santa Elena,0.0


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18954132 entries, 0 to 18954131
Data columns (total 7 columns):
 #   Column           Dtype         
---  ------           -----         
 0   event_timestamp  datetime64[ns]
 1   location_id      string        
 2   item_id          string        
 3   quantity         float32       
 4   cluster          int8          
 5   region           object        
 6   is_holiday       float32       
dtypes: datetime64[ns](1), float32(2), int8(1), object(1), string(2)
memory usage: 741.1+ MB


In [4]:
if "event_timestamp" in df.columns:
    df["event_timestamp"] = pd.to_datetime(df["event_timestamp"], errors="raise")

df = df.rename(columns={"event_timestamp": "datetime"})
df = df.sort_values(by='datetime')
df["quantity"] = df["quantity"].astype("float64")
df

,datetime,location_id,item_id,quantity,cluster,region,is_holiday
0,2017-01-01,25,99197,1.0,1,Santa Elena,0.0
10,2017-01-01,25,114790,2.0,1,Santa Elena,0.0
846,2017-01-01,25,1214325,7.0,1,Santa Elena,0.0
1,2017-01-01,25,103665,7.0,1,Santa Elena,0.0
2,2017-01-01,25,105574,1.0,1,Santa Elena,0.0
...,...,...,...,...,...,...,...
18954123,2017-06-30,54,2084278,7.0,3,Manabi,0.0
18954124,2017-06-30,54,2084557,3.0,3,Manabi,0.0
18954125,2017-06-30,54,2086882,2.0,3,Manabi,0.0
18954115,2017-06-30,54,2060793,14.0,3,Manabi,0.0


In [6]:
df_daily = (
    df.groupby("datetime", as_index=False)['quantity']
    .sum()
    .rename(columns={"quantity": "total_sales"})
)

In [7]:
df_daily

,datetime,total_sales
0,2017-01-01,1.208250e+04
1,2017-01-02,1.402305e+06
2,2017-01-03,1.104377e+06
3,2017-01-04,9.900935e+05
4,2017-01-05,7.776210e+05
...,...,...
176,2017-06-26,7.553678e+05
177,2017-06-27,6.822185e+05
178,2017-06-28,7.318970e+05
179,2017-06-29,6.308118e+05


In [8]:
df_daily = df_daily.sort_values(by='datetime')

In [9]:
df_daily

,datetime,total_sales
0,2017-01-01,1.208250e+04
1,2017-01-02,1.402305e+06
2,2017-01-03,1.104377e+06
3,2017-01-04,9.900935e+05
4,2017-01-05,7.776210e+05
...,...,...
176,2017-06-26,7.553678e+05
177,2017-06-27,6.822185e+05
178,2017-06-28,7.318970e+05
179,2017-06-29,6.308118e+05


In [ ]:
df_daily = df_daily.set_index("datetime").asfreq("D")
df_daily["total_sales"] = df_daily["total_sales"].fillna(0)
df_daily = df_daily.reset_index()
df_daily

,datetime,total_sales
0,2017-01-01,1.208250e+04
1,2017-01-02,1.402305e+06
2,2017-01-03,1.104377e+06
3,2017-01-04,9.900935e+05
4,2017-01-05,7.776210e+05
...,...,...
176,2017-06-26,7.553678e+05
177,2017-06-27,6.822185e+05
178,2017-06-28,7.318970e+05
179,2017-06-29,6.308118e+05


In [11]:
df_daily["day_of_week"] = df_daily["datetime"].dt.dayofweek
df_daily["month"] = df_daily["datetime"].dt.month
df_daily["is_weekend"] = df_daily["day_of_week"].isin([5, 6]).astype(int)

In [12]:
df_daily

,datetime,total_sales,day_of_week,month,is_weekend
0,2017-01-01,1.208250e+04,6,1,1
1,2017-01-02,1.402305e+06,0,1,0
2,2017-01-03,1.104377e+06,1,1,0
3,2017-01-04,9.900935e+05,2,1,0
4,2017-01-05,7.776210e+05,3,1,0
...,...,...,...,...,...
176,2017-06-26,7.553678e+05,0,6,0
177,2017-06-27,6.822185e+05,1,6,0
178,2017-06-28,7.318970e+05,2,6,0
179,2017-06-29,6.308118e+05,3,6,0


In [13]:
df_daily["lag_1"] = df_daily["total_sales"].shift(1)
df_daily["lag_7"] = df_daily["total_sales"].shift(7)
df_daily

,datetime,total_sales,day_of_week,month,is_weekend,lag_1,lag_7
0,2017-01-01,1.208250e+04,6,1,1,NaN,NaN
1,2017-01-02,1.402305e+06,0,1,0,1.208250e+04,NaN
2,2017-01-03,1.104377e+06,1,1,0,1.402305e+06,NaN
3,2017-01-04,9.900935e+05,2,1,0,1.104377e+06,NaN
4,2017-01-05,7.776210e+05,3,1,0,9.900935e+05,NaN
...,...,...,...,...,...,...,...
176,2017-06-26,7.553678e+05,0,6,0,1.092611e+06,791146.393970
177,2017-06-27,6.822185e+05,1,6,0,7.553678e+05,787326.717022
178,2017-06-28,7.318970e+05,2,6,0,6.822185e+05,766159.011011
179,2017-06-29,6.308118e+05,3,6,0,7.318970e+05,609857.830973


In [14]:
df_daily["rolling_mean_7"] = df_daily["total_sales"].rolling(7).mean()
df_daily

,datetime,total_sales,day_of_week,month,is_weekend,lag_1,lag_7,rolling_mean_7
0,2017-01-01,1.208250e+04,6,1,1,NaN,NaN,NaN
1,2017-01-02,1.402305e+06,0,1,0,1.208250e+04,NaN,NaN
2,2017-01-03,1.104377e+06,1,1,0,1.402305e+06,NaN,NaN
3,2017-01-04,9.900935e+05,2,1,0,1.104377e+06,NaN,NaN
4,2017-01-05,7.776210e+05,3,1,0,9.900935e+05,NaN,NaN
...,...,...,...,...,...,...,...,...
176,2017-06-26,7.553678e+05,0,6,0,1.092611e+06,791146.393970,819377.176525
177,2017-06-27,6.822185e+05,1,6,0,7.553678e+05,787326.717022,804361.712531
178,2017-06-28,7.318970e+05,2,6,0,6.822185e+05,766159.011011,799467.137370
179,2017-06-29,6.308118e+05,3,6,0,7.318970e+05,609857.830973,802460.561946


In [15]:
df_daily = df_daily.dropna().reset_index(drop=True)
df_daily

,datetime,total_sales,day_of_week,month,is_weekend,lag_1,lag_7,rolling_mean_7
0,2017-01-08,1.192544e+06,6,1,1,1.103806e+06,1.208250e+04,1.058621e+06
1,2017-01-09,7.936869e+05,0,1,0,1.192544e+06,1.402305e+06,9.716755e+05
2,2017-01-10,7.426356e+05,1,1,0,7.936869e+05,1.104377e+06,9.199982e+05
3,2017-01-11,7.707427e+05,2,1,0,7.426356e+05,9.900935e+05,8.886623e+05
4,2017-01-12,6.296358e+05,3,1,0,7.707427e+05,7.776210e+05,8.675216e+05
...,...,...,...,...,...,...,...,...
169,2017-06-26,7.553678e+05,0,6,0,1.092611e+06,7.911464e+05,8.193772e+05
170,2017-06-27,6.822185e+05,1,6,0,7.553678e+05,7.873267e+05,8.043617e+05
171,2017-06-28,7.318970e+05,2,6,0,6.822185e+05,7.661590e+05,7.994671e+05
172,2017-06-29,6.308118e+05,3,6,0,7.318970e+05,6.098578e+05,8.024606e+05


In [16]:
df_daily

,datetime,total_sales,day_of_week,month,is_weekend,lag_1,lag_7,rolling_mean_7
0,2017-01-08,1.192544e+06,6,1,1,1.103806e+06,1.208250e+04,1.058621e+06
1,2017-01-09,7.936869e+05,0,1,0,1.192544e+06,1.402305e+06,9.716755e+05
2,2017-01-10,7.426356e+05,1,1,0,7.936869e+05,1.104377e+06,9.199982e+05
3,2017-01-11,7.707427e+05,2,1,0,7.426356e+05,9.900935e+05,8.886623e+05
4,2017-01-12,6.296358e+05,3,1,0,7.707427e+05,7.776210e+05,8.675216e+05
...,...,...,...,...,...,...,...,...
169,2017-06-26,7.553678e+05,0,6,0,1.092611e+06,7.911464e+05,8.193772e+05
170,2017-06-27,6.822185e+05,1,6,0,7.553678e+05,7.873267e+05,8.043617e+05
171,2017-06-28,7.318970e+05,2,6,0,6.822185e+05,7.661590e+05,7.994671e+05
172,2017-06-29,6.308118e+05,3,6,0,7.318970e+05,6.098578e+05,8.024606e+05


In [17]:
df_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174 entries, 0 to 173
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   datetime        174 non-null    datetime64[ns]
 1   total_sales     174 non-null    float64       
 2   day_of_week     174 non-null    int32         
 3   month           174 non-null    int32         
 4   is_weekend      174 non-null    int64         
 5   lag_1           174 non-null    float64       
 6   lag_7           174 non-null    float64       
 7   rolling_mean_7  174 non-null    float64       
dtypes: datetime64[ns](1), float64(4), int32(2), int64(1)
memory usage: 9.6 KB


In [ ]:
df_daily["total_sales"] = df_daily["total_sales"].astype("float64")
df_daily

In [ ]:
df_daily.info()